# Modelo de Traslado de Valores - FINABIEN
## 01. Limpieza de Datos
### El propósito de este 'notebook' es generar la base de datos con las características necesarias para el Modelo de Traslado de Valores - FINABIEN.

In [1]:
import os
import pandas as pd
import pickle
import Funciones as f
import holidays

In [2]:
# Leer el CSV, especificando el nombre de la columna correcta
festivos = pd.read_csv(
    "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV/2. Datos/festivos.csv",
    parse_dates=["no_laborales"])

In [ ]:
ruta = 'C:/Users/jose.luis.flores/Documents/Repositorios/MTDV/2. Datos' # Cambia esto por la ruta a tu carpeta
df = f.cargar_txts_en_dataframe(ruta)
df.head()

In [ ]:
print(df.shape)
print(df.dtypes)

In [5]:
# Lista de nombres de estados en orden
nombres_estados = [
    'aguascalientes',
    'baja_california',
    'baja_california_sur',
    'campeche',
    'coahuila',
    'colima',
    'chiapas',
    'chihuahua',
    'cdmx',
    'durango',
    'guanajuato',
    'guerrero',
    'hidalgo',
    'jalisco',
    'mexico',
    'michoacan',
    'morelos',
    'nayarit',
    'nuevo_leon',
    'oaxaca',
    'puebla',
    'queretaro',
    'quintana_roo',
    'san_luis_potosi',
    'sinaloa',
    'sonora',
    'tabasco',
    'tamaulipas',
    'tlaxcala',
    'veracruz',
    'yucatan',
    'zacatecas'
]

# Diccionario de mapeo estado -> nombre
estado_a_nombre = {i + 1: nombre for i, nombre in enumerate(nombres_estados)}

# Asignar la nueva columna directamente
df['nombre_estado'] = df['estado'].map(estado_a_nombre)

In [ ]:
df.head()

In [ ]:
df_mod = f.transformar_dataframe(df)
df_mod.head()

In [ ]:
df_mod.dtypes

In [9]:
# Crear copia del DataFrame original
df_mod = df_mod.copy()

# 1. Crear columna 'entradas'
df_mod['entradas'] = df_mod[
    ['corresponsalia_entrada', 'trans_int_entrada', 'giro_nac_expedicion', 'terceros_entrada']
].sum(axis=1)

# 2. Crear columna 'salidas'
df_mod['salidas'] = df_mod[
    ['corresponsalia_salida', 'trans_int_salida', 'giro_nac_pago', 'terceros_salida', 'grandes_usuarios']
].sum(axis=1)

# 3. Crear columna 'flujo_efectivo'
df_mod['flujo_efectivo'] = df_mod['entradas'] - df_mod['salidas']

# 4. Eliminar columnas utilizadas
df_mod.drop(columns=[
    'corresponsalia_entrada', 'trans_int_entrada', 'giro_nac_expedicion', 'terceros_entrada', 
    'corresponsalia_salida', 'trans_int_salida', 'giro_nac_pago', 'terceros_salida', 'grandes_usuarios'
], inplace=True)

In [ ]:
df_mod.head()

In [11]:
# Definir las columnas prioritarias
columnas_principales = ['fecha', 'nombre_estado', 'id_sucursal', 'entradas', 'salidas', 'flujo_efectivo']

# Obtener las demás columnas que no están en las principales
otras_columnas = [col for col in df_mod.columns if col not in columnas_principales]

# Reordenar el DataFrame
df_mod = df_mod[columnas_principales + otras_columnas]

In [ ]:
df_mod.head()

In [13]:
# Ordenar los datos por id_sucursal y fecha
df_mod = df_mod.sort_values(by=['id_sucursal', 'fecha']).reset_index(drop=True)

In [ ]:
# Verificar que días trabajan las sucursales
dias_semana = df_mod['fecha'].dt.dayofweek  # Lunes=0, Domingo=6
print("Días de la semana presentes:", dias_semana.unique())  

In [15]:
# Obtener años únicos presentes en tu DataFrame
years = df_mod['fecha'].dt.year.unique()

# Generar festivos oficiales de México para esos años
festivos_mx = holidays.MX(years=years)

In [ ]:
# Suponiendo que tu DataFrame principal tiene una columna 'fecha'
df_mod["no_laborales"] = df_mod["fecha"].isin(festivos["no_laborales"]).astype(int)

df_mod['festivo_mx'] = df_mod['fecha'].isin(festivos_mx).astype(int)

In [ ]:
df_mod.head()

In [18]:
# Día de la semana (0=Lunes, 4=Viernes)
df_mod['dia_semana'] = df_mod['fecha'].dt.dayofweek

# Semana del año
df_mod['semana_año'] = df_mod['fecha'].dt.isocalendar().week

# Mes 
df_mod['mes'] = df_mod['fecha'].dt.month

#Año
df_mod['año'] = df_mod['fecha'].dt.year

In [26]:
# Esta es la parte que se modifico
# Paso 1: Crear columna de año y trimestre fijo
def asignar_trimestre(mes):
    if mes <= 3:
        return 1
    elif mes <= 6:
        return 2
    elif mes <= 9:
        return 3
    else:
        return 4

df_mod['año'] = df_mod['fecha'].dt.year
df_mod['trimestre'] = df_mod['fecha'].dt.month.apply(asignar_trimestre)

# Paso 2: Calcular el promedio del flujo por sucursal y trimestre
promedios = (
    df_mod.groupby(['id_sucursal', 'año', 'trimestre'])['flujo_efectivo']
    .mean()
    .reset_index()
    .rename(columns={'flujo_efectivo': 'flujo_promedio_trimestral'})
)

# Paso 3: Clasificar como captadora (1) si el flujo promedio es positivo
promedios['es_captadora'] = (promedios['flujo_promedio_trimestral'] > 0).astype(int)

# Paso 4: Unir esta clasificación de vuelta al dataframe original
df_mod = df_mod.merge(
    promedios[['id_sucursal', 'año', 'trimestre', 'es_captadora']],
    on=['id_sucursal', 'año', 'trimestre'],
    how='left'
)

In [20]:
# Media móvil de 3, 5, 10 y 14 días laborales
df_mod['media_movil_3'] = df_mod['flujo_efectivo'].rolling(3).mean()
df_mod['media_movil_5'] = df_mod['flujo_efectivo'].rolling(5).mean()
df_mod['media_movil_10'] = df_mod['flujo_efectivo'].rolling(10).mean()
df_mod['media_movil_14'] = df_mod['flujo_efectivo'].rolling(14).mean()

In [21]:
# Rezagos de 1, 2, 3 y 5 días laborales
for lag in [1, 2, 3, 5]:
    df_mod[f'lag_{lag}'] = df_mod['flujo_efectivo'].shift(lag)

In [22]:
# Eliminar filas con al menos un NaN en cualquier columna
df_mod = df_mod.dropna(how='any')

# Verificar resultado
print(f"Filas después de eliminar NaN: {len(df_mod)}")

Filas después de eliminar NaN: 903811


In [ ]:
df_mod.head(10)

In [ ]:
df_mod.dtypes

In [ ]:
# Ruta donde se guardará el archivo
ruta_archivo_flujo_caja = "C:/Users/jose.luis.flores/Documents/Repositorios/MTDV/2. Datos/df_mod.pkl"

# Guardar el DataFrame en un archivo pickle
with open(ruta_archivo_flujo_caja, "wb") as archivo:
    pickle.dump(df_mod, archivo)

print(f"Base de datos guardada en {ruta_archivo_flujo_caja}")